# TP : Implémentation de l'Algorithme d'Estimation de Phase Quantique (QPE)

L'algorithme d'estimation de phase quantique (QPE) permet d'estimer la phase $\theta$ d'une valeur propre associée à un vecteur propre d'un opérateur unitaire $U$ tel que : 
$$U|\psi\rangle = e^{2\pi i \theta}|\psi\rangle$$

**Objectif de cette première partie :**
Vous allez construire pas à pas la classe `MonQPE`. Votre travail consiste à compléter les méthodes individuelles permettant de construire les différentes parties du circuit quantique. La méthode `run` (déjà implémentée) se chargera d'assembler votre travail, d'afficher le circuit proprement et de l'exécuter.

In [ ]:
# Exécutez cette cellule pour importer les librairies nécessaires
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import QFT
from qiskit.primitives import StatevectorSampler as Sampler
from IPython.display import display

## 1. Complétez la construction du circuit

Vous devez compléter les méthodes suivantes dans la classe ci-dessous :
1. **`apply_initial_state`** : Prépare l'état propre $|\psi\rangle$ sur le registre principal (cible).
2. **`apply_hadamard`** : Applique une porte Hadamard sur tous les qubits du registre d'ancilla (pour créer la superposition).
3. **`apply_controlled_unitary`** : Applique les opérations $U^{2^j}$ contrôlées par le $j$-ème qubit d'ancilla. *Indice : Utilisez la méthode `.control()` sur votre porte unitaire et ajoutez-la au circuit avec `.append()`*.
4. **`apply_inverse_qft`** : Applique la transformée de Fourier quantique inverse. *Indice : Vous pouvez utiliser `QFT(num_qubits, inverse=True).to_instruction()`*.
5. **`measure`** : Mesure le registre d'ancilla dans le registre classique.

In [ ]:
class MonQPE:
    def __init__(self, unitary_gate, state_prep_circuit, num_ancilla):
        """
        Initialise l'algorithme QPE.
        :param unitary_gate: La porte quantique U (objet Gate de Qiskit).
        :param state_prep_circuit: Circuit préparant l'état propre |\psi>.
        :param num_ancilla: Nombre de qubits pour le registre de comptage (précision).
        """
        self.unitary = unitary_gate
        self.state_prep = state_prep_circuit
        self.num_ancilla = num_ancilla
        self.num_target = state_prep_circuit.num_qubits
        
        # Création des registres
        self.anc_register = QuantumRegister(self.num_ancilla, 'ancilla')
        self.target_register = QuantumRegister(self.num_target, 'target')
        self.classical_register = ClassicalRegister(self.num_ancilla, 'meas')
        
        # Initialisation du circuit global
        self.qc = QuantumCircuit(self.anc_register, self.target_register, self.classical_register)

    def apply_initial_state(self):
        """ ÉTAPE 1 : Applique la préparation de l'état sur le registre cible. """
        # --- À COMPLÉTER ---
        # Indice : utilisez self.qc.compose(...)
        pass
        
    def apply_hadamard(self):
        """ ÉTAPE 2 : Applique des portes H sur le registre d'ancilla. """
        # --- À COMPLÉTER ---
        pass

    def apply_controlled_unitary(self):
        """ ÉTAPE 3 : Applique les portes U contrôlées (U^{2^j}). """
        # --- À COMPLÉTER ---
        # Pour chaque qubit d'ancilla j de 0 à num_ancilla-1 :
        # 1. Créez la porte U à la puissance 2^j (ex: self.unitary.power(2**j))
        # 2. Transformez-la en porte contrôlée
        # 3. Ajoutez-la au circuit (self.qc.append(...))
        pass

    def apply_inverse_qft(self):
        """ ÉTAPE 4 : Applique la Transformée de Fourier Quantique Inverse. """
        # --- À COMPLÉTER ---
        pass

    def measure(self):
        """ ÉTAPE 5 : Mesure le registre d'ancilla. """
        # --- À COMPLÉTER ---
        pass

    def run(self, shots=1024):
        """ 
        MÉTHODE FOURNIE : Assemble le circuit, l'affiche et l'exécute.
        """
        # 1. Construction du circuit en appelant vos méthodes
        self.apply_initial_state()
        self.qc.barrier() # Barrière visuelle
        
        self.apply_hadamard()
        self.qc.barrier()
        
        self.apply_controlled_unitary()
        self.qc.barrier()
        
        self.apply_inverse_qft()
        self.qc.barrier()
        
        self.measure()

        # 2. Affichage graphique du circuit
        print("Circuit QPE généré :")
        # L'utilisation de to_instruction() ou .control() dans vos méthodes 
        # garantit que U apparaît comme un bloc unique !
        display(self.qc.draw('mpl', fold=-1))

        # 3. Exécution avec le Sampler
        print("\nExécution en cours...")
        sampler = Sampler()
        job = sampler.run([self.qc], shots=shots)        
        sampler_results = job.result()
        
        cr_name = list(sampler_results[0].data.keys())[0]
        counts = sampler_results[0].data[cr_name].get_counts()
        
        print("Résultats bruts (Probabilités) :", counts)
        return counts

## 2. Phase de Test

Pour vérifier que votre classe fonctionne, nous allons définir un cas d'usage simple :
- La porte unitaire $U$ sera une porte de Phase quantique $P(\theta)$ avec $\theta = \pi/4$.
- En unités de $2\pi$, la phase attendue est donc $0.125$ (soit `001` en binaire pour 3 ancillas).
- L'état propre associé à cette porte est l'état $|1\rangle$ (on appliquera donc une porte $X$ pour la préparation d'état).

In [ ]:
from qiskit.circuit.library import PhaseGate

# 1. Définition de l'opérateur U (Porte de Phase)
# La valeur propre associée à |1> sera exp(i * pi/4)
# La phase est donc theta = (pi/4) / (2*pi) = 1/8 = 0.125
theta_angle = np.pi / 4
U_gate = PhaseGate(theta_angle)
# On donne un nom explicite pour l'affichage du bloc
U_gate.name = "U(pi/4)"

# 2. Préparation de l'état propre |1> (Porte X)
prep_circuit = QuantumCircuit(1)
prep_circuit.x(0)

# 3. Création de l'instance de votre classe (Précision : 3 qubits)
qpe_instance = MonQPE(unitary_gate=U_gate, state_prep_circuit=prep_circuit, num_ancilla=3)

# 4. Exécution (construit, affiche et simule)
counts = qpe_instance.run(shots=1000)

# Note pour la lecture des résultats : 
# L'état binaire mesuré le plus probable devrait être '001'.
# En binaire fractionnaire, 0.001 = 1/8 = 0.125 !

In [ ]:
from qiskit.circuit.library import PhaseGate

# 1. Définition de l'opérateur U (Porte de Phase)
# La valeur propre associée à |1> sera exp(i * pi/4)
# La phase est donc theta = (pi/4) / (2*pi) = 1/8 = 0.125
theta_angle = np.pi / 4
U_gate = PhaseGate(theta_angle)
# On donne un nom explicite pour l'affichage du bloc
U_gate.name = "U(pi/4)"

# 2. Préparation de l'état propre |1> (Porte X)
prep_circuit = QuantumCircuit(1)
prep_circuit.x(0)

# 3. Création de l'instance de votre classe (Précision : 3 qubits)
qpe_instance = MonQPE(unitary_gate=U_gate, state_prep_circuit=prep_circuit, num_ancilla=3)

# 4. Exécution (construit, affiche et simule)
counts = qpe_instance.run(shots=1000)

# Note pour la lecture des résultats : 
# L'état binaire mesuré le plus probable devrait être '001'.
# En binaire fractionnaire, 0.001 = 1/8 = 0.125 !

## 3. Ce que nous avons accompli

Si tout s'est bien passé, vous avez vu apparaître un circuit où les opérations unitaires contrôlées s'affichent sous forme de blocs bien définis (`cU(pi/4)`), et le résultat le plus probable correspond exactement à la phase théorique attendue.

**Prochaines étapes (à suivre) :**
Dans la suite de ce TP, nous rajouterons de nouvelles fonctionnalités à notre classe. Nous aborderons :
1. Le post-traitement (convertir automatiquement la chaîne binaire `001` en décimal `0.125`).
2. La gestion du bruit et des phases qui ne tombent pas "juste" (phases non représentables exactement avec le nombre de qubits d'ancilla choisi).
3. L'optimisation et le décompte des portes pour estimer la profondeur réelle du circuit sur un processeur quantique.

# Partie 2 : Application à la Chimie Quantique (Molécule $H_2$)

Maintenant que notre algorithme QPE est fonctionnel, nous allons l'utiliser pour résoudre un vrai problème physique : estimer l'énergie de l'état fondamental d'une molécule.

**Le workflow de la chimie quantique sur ordinateur quantique :**
1. **Le Driver (PySCF)** : Calcule les intégrales moléculaires classiques (physique classique) pour définir la molécule.
2. **Le Hamiltonien Fermionique** : Représente l'énergie des électrons.
3. **Le Mapping (Jordan-Wigner)** : Convertit les électrons (fermions) en qubits simulables.
4. **L'État Initial** : Pour que le QPE trouve l'énergie de l'état fondamental, il faut lui fournir une bonne approximation initiale. Nous utiliserons l'état de **Hartree-Fock**.
5. **L'Évolution Temporelle (Trotterisation)** : Le QPE a besoin d'un opérateur unitaire $U$. Nous allons transformer notre Hamiltonien $H$ en un opérateur unitaire via l'équation de Schrödinger : $U = e^{-iHt}$.

In [ ]:
# Installation requise si non présente : !pip install qiskit-nature pyscf
import numpy as np
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter
from qiskit import QuantumCircuit

## 1. Modélisation de la molécule et Mapping

Dans cette étape, nous allons définir la molécule $H_2$ à sa distance d'équilibre (environ 0.735 Ångströms). Le driver `PySCF` va calculer les propriétés de la molécule et nous allons extraire le **Hamiltonien**.

Puisque les ordinateurs quantiques utilisent des qubits (et non des électrons), nous devons utiliser un "Mapper" (comme la transformation de Jordan-Wigner) pour convertir notre Hamiltonien fermionique en une somme d'opérateurs de Pauli (X, Y, Z).

**À vous de jouer :** Complétez le code pour générer l'opérateur qubit.

In [ ]:
# 1. Configuration du driver PySCF pour la molécule H2
driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.735",
    basis="sto3g",
    charge=0,
    spin=0,
)

# 2. Exécution du driver pour obtenir le problème électronique
problem = driver.run()
hamiltonian = problem.hamiltonian.second_q_op()

print("Fermionic Hamiltonian:\n", hamiltonian)

# --- À COMPLÉTER : Mapping vers des Qubits ---
# 1. Instanciez le JordanWignerMapper()
mapper = # <VOTRE CODE ICI>

# 2. Appliquez la méthode .map() du mapper sur notre 'hamiltonian'
qubit_op = # <VOTRE CODE ICI>

print("\nQubit Hamiltonian (Opérateurs de Pauli):")
print(qubit_op)

## 2. Préparation pour le QPE (Hartree-Fock et Trotterisation)

Pour utiliser `MonQPE`, nous avons besoin de deux choses :
1. **Un circuit de préparation d'état (`state_prep_circuit`)** : Nous allons générer le circuit qui crée l'état de Hartree-Fock (la meilleure approximation classique de l'état fondamental).
2. **Une porte unitaire (`unitary_gate`)** : Nous devons transformer notre Hamiltonien $H$ en $U = e^{-iHt}$. 
   Comme $H$ est une somme d'opérateurs qui ne commutent pas forcément, on utilise la formule de **Trotter-Suzuki** pour approximer l'exponentielle. Qiskit propose la `PauliEvolutionGate` pour faire cela automatiquement !

In [ ]:
# --- À COMPLÉTER : État de Hartree-Fock ---
# Qiskit Nature propose une classe toute prête : HartreeFock(num_spatial_orbitals, num_particles, mapper)
num_spatial_orbitals = problem.num_spatial_orbitals
num_particles = problem.num_particles

# 1. Instanciez l'état de Hartree-Fock
hf_circuit = # <VOTRE CODE ICI>

print("Circuit de préparation de l'état de Hartree-Fock :")
display(hf_circuit.draw('mpl'))

# --- À COMPLÉTER : L'Opérateur Unitaire (Évolution temporelle) ---
# Le QPE va estimer la phase de U = e^{-i H t}. 
# Pour l'exercice, nous fixons le temps d'évolution t = 1.0 (unité arbitraire ici).
evolution_time = 1.0

# 2. Créez la porte d'évolution en utilisant PauliEvolutionGate(operator, time, synthesis)
# Utilisez LieTrotter(reps=1) pour la synthèse (synthesis).
unitary_gate = # <VOTRE CODE ICI>
unitary_gate.name = "U(H2)"

## 3. Exécution du QPE sur la molécule $H_2$

Tout est prêt ! Nous avons notre circuit de préparation (Hartree-Fock) et notre porte unitaire (l'évolution temporelle du Hamiltonien de $H_2$).

Il ne reste plus qu'à instancier la classe `MonQPE` que vous avez codée dans la partie 1 et à analyser les résultats. Notez que le registre cible nécessite maintenant 4 qubits (car la molécule $H_2$ en base STO-3G nécessite 4 spin-orbitales).

*Remarque sur le temps de calcul : Le circuit généré par la Trotterisation est beaucoup plus profond que notre simple porte de phase précédente. L'affichage du circuit complet peut être très imposant !*

In [ ]:
# On choisit 4 qubits d'ancilla pour avoir une précision correcte
num_ancilla_qubits = 4

# On utilise la classe MonQPE définie dans la Partie 1
qpe_h2 = MonQPE(
    unitary_gate=unitary_gate,
    state_prep_circuit=hf_circuit,
    num_ancilla=num_ancilla_qubits
)

print(f"Nombre total de qubits du circuit : {qpe_h2.qc.num_qubits}")

# Exécution !
# (L'affichage du circuit va montrer le bloc "U(H2)" contrôlé par les ancillas)
counts_h2 = qpe_h2.run(shots=2000)

# Extraction de la chaîne binaire la plus mesurée
max_binary = max(counts_h2, key=counts_h2.get)
print(f"\nL'état mesuré le plus probable est : {max_binary} avec une probabilité de {counts_h2[max_binary]:.2f}")

## 4. Analyse des résultats (De la Phase à l'Énergie)

L'algorithme QPE nous retourne une chaîne binaire représentant une phase $\theta \in [0, 1[$.
En binaire fractionnaire, une chaîne $q_1 q_2 q_3 q_4$ correspond à la valeur décimale :
$$ \theta = \frac{q_1}{2} + \frac{q_2}{4} + \frac{q_3}{8} + \frac{q_4}{16} $$

**Le lien avec l'énergie :**
L'opérateur était $U = e^{-iHt}$. Ses valeurs propres sont de la forme $e^{2i\pi\theta}$.
On a donc la relation : $-E \cdot t = 2\pi\theta \pmod{2\pi}$

*Note pédagogique* : Dans un vrai cas d'usage, il faut effectuer un "décalage" (shift) du Hamiltonien pour s'assurer que toutes les énergies sont positives et rentrent parfaitement dans l'intervalle $[0, 2\pi[$. Ici, la lecture de l'énergie exacte demanderait de borner les spectres, mais la mécanique d'extraction de la phase reste la même !

In [ ]:
def binary_fraction_to_decimal(binary_string):
    """ Convertit une chaîne binaire fractionnaire en nombre décimal. """
    decimal_value = 0
    for i, bit in enumerate(binary_string):
        decimal_value += int(bit) * (2 ** -(i + 1))
    return decimal_value

# Calcul de la phase theta
theta_estimate = binary_fraction_to_decimal(max_binary)
print(f"La phase quantique estimée est : theta = {theta_estimate}")

# L'énergie (non normalisée) correspondante peut en être déduite
# E = - 2 * pi * theta / t 
energy_estimate = - 2 * np.pi * theta_estimate / evolution_time

print(f"L'énergie associée (modulo le repliement de phase) est de : {energy_estimate:.4f} Hartree")

## 5. Comparaison avec les résultats exacts

Pour savoir si notre QPE a bien fonctionné, nous devons calculer l'énergie exacte de la molécule. Sur un simulateur classique, nous pouvons simplement représenter notre Hamiltonien (`qubit_op`) sous forme de matrice et trouver sa plus petite valeur propre (Exact Diagonalization).

Qiskit propose pour cela le `NumPyMinimumEigensolver`.

**Attention à deux subtilités de la chimie quantique :**
1. **L'énergie nucléaire** : Le Hamiltonien simulé sur les qubits ne calcule que l'énergie des électrons (Énergie électronique). L'énergie totale de la molécule nécessite d'y ajouter la répulsion entre les noyaux, qui est une constante classique.
2. **Les énergies négatives** : L'énergie de l'état fondamental de $H_2$ est négative. Or, l'algorithme QPE retourne une phase $\theta \in [0, 1[$. 
   Puisque $e^{-iEt} = e^{2i\pi\theta}$, une énergie négative $E < 0$ va "s'enrouler" autour du cercle unitaire et donner une phase $\theta$ proche de 1 (ex: $0.85$). 
   Si $\theta > 0.5$, la vraie énergie est alors : $E = \frac{2\pi(\theta - 1)}{t}$.

In [ ]:
# Installation requise si non présente : !pip install qiskit-algorithms
from qiskit_algorithms import NumPyMinimumEigensolver

# 1. Calcul classique exact (Diagonalisation de la matrice du Hamiltonien)
exact_solver = NumPyMinimumEigensolver()
exact_result = exact_solver.compute_minimum_eigenvalue(qubit_op)

# L'énergie retournée par le solveur est l'énergie électronique pure
exact_electronic_energy = exact_result.eigenvalue.real

# On y ajoute la constante de répulsion nucléaire calculée par PySCF
nuclear_repulsion = problem.nuclear_repulsion_energy
exact_total_energy = exact_electronic_energy + nuclear_repulsion

print(f"--- RÉSULTATS EXACTS (Classiques) ---")
print(f"Énergie électronique exacte : {exact_electronic_energy:.4f} Hartree")
print(f"Répulsion nucléaire         : {nuclear_repulsion:.4f} Hartree")
print(f"Énergie TOTALE exacte       : {exact_total_energy:.4f} Hartree\n")

## 6. Correction de la phase QPE et Bilan

Maintenant que nous connaissons la théorie sur les énergies négatives, nous allons corriger notre calcul d'énergie issu du QPE et le comparer au résultat exact. 

Plus le nombre de qubits d'ancilla (`num_ancilla_qubits`) est grand, plus notre estimation QPE devrait s'approcher du résultat de `NumPyMinimumEigensolver`.

In [ ]:
# 1. Correction du repliement de phase (si theta > 0.5, l'énergie était négative)
if theta_estimate > 0.5:
    corrected_theta = theta_estimate - 1.0
else:
    corrected_theta = theta_estimate

# 2. Recalcul de l'énergie électronique estimée par le QPE
# Formule : E * t = - 2 * pi * theta  =>  E = - 2 * pi * theta / t
qpe_electronic_energy = - 2 * np.pi * corrected_theta / evolution_time

# 3. Ajout de la répulsion nucléaire pour obtenir l'énergie totale
qpe_total_energy = qpe_electronic_energy + nuclear_repulsion

# 4. Affichage et calcul de l'erreur
print(f"--- RÉSULTATS QPE ---")
print(f"Phase brute lue (theta)       : {theta_estimate}")
print(f"Phase corrigée (E négative)   : {corrected_theta}")
print(f"Énergie électronique estimée  : {qpe_electronic_energy:.4f} Hartree")
print(f"Énergie TOTALE estimée (QPE)  : {qpe_total_energy:.4f} Hartree\n")

print(f"--- COMPARAISON ---")
erreur_absolue = abs(exact_total_energy - qpe_total_energy)
print(f"Énergie QPE   : {qpe_total_energy:.4f} Hartree")
print(f"Énergie Exacte: {exact_total_energy:.4f} Hartree")
print(f"Erreur absolue: {erreur_absolue:.4f} Hartree")

# Petit challenge pour les étudiants :
print("\n💡 Exercice : Modifiez 'num_ancilla_qubits' dans la cellule d'exécution (essayez 5 ou 6).")
print("Comment évolue l'erreur absolue ? (Attention au temps de calcul !)")